# Myotome Downsampling Mesh Similarity

Evaluate bilateral myotome surface reconstruction and shape similarity across retained-cell fractions.

This curated notebook targets the current Dynamo-free Spateo API. Edit the path/configuration cells for a new system before execution.


In [ ]:
import numpy as np

import spateo as st


# Surface reconstruction


In [ ]:
cpo1 = [
    (7235.672822135923, -9333.743725966886, 29537.530999873827),
    (5265.3827, 317.52565000000004, 1200.0),
    (-0.9910735612049205, -0.13108990476291582, 0.024261763123203748),
]

cpo2 = [
    (-1508.524414183453, -10974.135117384183, 25563.352608545407),
    (4720.1671, 112.69174999999996, 1200.0),
    (-0.6800775362520292, 0.7171120242306434, 0.15246274754575018),
]

cpo3 = [
    (-134.8569326485603, -22618.34076227472, 15863.43005107714),
    (4720.1671, 112.69174999999996, 1200.0),
    (-0.9734993446053986, 0.06684381746631253, -0.21870283518827516),
]

cpo4 = [
    (-5006.789177572386, -25557.216094653202, -115.0783686672523),
    (4720.1671, 112.69174999999996, 1200.0),
    (-0.9334155956236307, 0.35598914126682585, -0.04480019136890938),
]


## Load and validate data


In [ ]:
Myotome_left = st.read_h5ad("/DATA/User/gaomohan/Myotome_v2_left.h5ad")
Myotome_right = st.read_h5ad("/DATA/User/gaomohan/Myotome_v1_right.h5ad")
Myotome_left, Myotome_right


## Construct the point-cloud model


In [ ]:
Myotome_left_pc, plot_cmap = st.tdr.construct_pc(
    adata=Myotome_left.copy(),
    spatial_key="spatial",
    groupby="celltype",
    key_added="tissue",
    colormap="#00BFC4",
)

Myotome_right_pc, plot_cmap = st.tdr.construct_pc(
    adata=Myotome_right.copy(),
    spatial_key="spatial",
    groupby="celltype",
    key_added="tissue",
    colormap="#00BFC4",
)


## Reconstruct the surface mesh


In [ ]:
Myotome_left_mesh, _, _ = st.tdr.construct_surface(
    pc=Myotome_left_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.4},
    smooth=5000,
    scale_factor=1.00,
)

Myotome_right_mesh, _, _ = st.tdr.construct_surface(
    pc=Myotome_right_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.6},
    smooth=5000,
    scale_factor=1.00,
)


In [ ]:
st.pl.three_d_plot(
    model=Myotome_left_mesh,
    key="tissue",
    model_style="surface",
    show_axes=True,
    jupyter="static",
    window_size=(800, 800),
    cpo=cpo1,
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([Myotome_left_mesh, Myotome_left_pc]),
    key="tissue",
    model_style=["surface", "points"],
    show_axes=True,
    jupyter="static",
    window_size=(800, 800),
    cpo=cpo1,
)


# Retain 10% of cells


In [ ]:
n_sample1 = 2329
n_sample2 = 2531

rng = np.random.default_rng(42)
idx1 = rng.choice(Myotome_left.n_obs, size=n_sample1, replace=False)
idx2 = rng.choice(Myotome_right.n_obs, size=n_sample2, replace=False)

Myotome_left_0_1 = Myotome_left[idx1].copy()
Myotome_right_0_1 = Myotome_right[idx2].copy()
Myotome_left_0_1, Myotome_right_0_1


In [ ]:
Myotome_left_0_1_pc, plot_cmap = st.tdr.construct_pc(
    adata=Myotome_left_0_1.copy(),
    spatial_key="spatial",
    groupby="celltype",
    key_added="tissue",
    colormap="#00BFC4",
)

Myotome_right_0_1_pc, plot_cmap = st.tdr.construct_pc(
    adata=Myotome_right_0_1.copy(),
    spatial_key="spatial",
    groupby="celltype",
    key_added="tissue",
    colormap="#00BFC4",
)


# Right-myotome mesh at 10% retained cells


In [ ]:
Myotome_right_0_1_mesh, _, _ = st.tdr.construct_surface(
    pc=Myotome_right_0_1_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.6},
    smooth=5000,
    scale_factor=1.00,
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([Myotome_right_0_1_mesh, Myotome_right_0_1_pc]),
    key="tissue",
    model_style=["surface", "points"],
    show_axes=True,
    jupyter="static",
    window_size=(800, 800),
    cpo=cpo1,
)


# Left-myotome mesh at 10% retained cells


In [ ]:
Myotome_left_0_1_mesh, _, _ = st.tdr.construct_surface(
    pc=Myotome_left_0_1_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.4},
    smooth=5000,
    scale_factor=1.00,
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([Myotome_left_0_1_mesh, Myotome_left_0_1_pc]),
    key="tissue",
    model_style=["surface", "points"],
    show_axes=True,
    jupyter="static",
    window_size=(800, 800),
    cpo=cpo1,
)


In [ ]:
score1 = st.tdr.pairwise_shape_similarity(
    model1_pcs=np.asarray(Myotome_left_mesh.points),
    model2_pcs=np.asarray(Myotome_left_0_1_mesh.points),
)
print(score1)

score2 = st.tdr.pairwise_shape_similarity(
    model1_pcs=np.asarray(Myotome_right_mesh.points),
    model2_pcs=np.asarray(Myotome_right_0_1_mesh.points),
)
print(score2)


# Retain 20% of cells


In [ ]:
n_sample1 = 4658
n_sample2 = 5062

rng = np.random.default_rng(42)
idx1 = rng.choice(Myotome_left.n_obs, size=n_sample1, replace=False)
idx2 = rng.choice(Myotome_right.n_obs, size=n_sample2, replace=False)

Myotome_left_0_2 = Myotome_left[idx1].copy()
Myotome_right_0_2 = Myotome_right[idx2].copy()
Myotome_left_0_2, Myotome_right_0_2


In [ ]:
Myotome_left_0_2_pc, plot_cmap = st.tdr.construct_pc(
    adata=Myotome_left_0_2.copy(),
    spatial_key="spatial",
    groupby="celltype",
    key_added="tissue",
    colormap="#00BFC4",
)

Myotome_right_0_2_pc, plot_cmap = st.tdr.construct_pc(
    adata=Myotome_right_0_2.copy(),
    spatial_key="spatial",
    groupby="celltype",
    key_added="tissue",
    colormap="#00BFC4",
)


# Right-myotome mesh at 20% retained cells


In [ ]:
Myotome_right_0_2_mesh, _, _ = st.tdr.construct_surface(
    pc=Myotome_right_0_2_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.6},
    smooth=5000,
    scale_factor=1.00,
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([Myotome_right_0_2_mesh, Myotome_right_0_2_pc]),
    key="tissue",
    model_style=["surface", "points"],
    show_axes=True,
    jupyter="static",
    window_size=(800, 800),
    cpo=cpo1,
)


# Left-myotome mesh at 20% retained cells


In [ ]:
Myotome_left_0_2_mesh, _, _ = st.tdr.construct_surface(
    pc=Myotome_left_0_2_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.4},
    smooth=5000,
    scale_factor=1.00,
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([Myotome_left_0_2_mesh, Myotome_left_0_2_pc]),
    key="tissue",
    model_style=["surface", "points"],
    show_axes=True,
    jupyter="static",
    window_size=(800, 800),
    cpo=cpo1,
)


In [ ]:
score1 = st.tdr.pairwise_shape_similarity(
    model1_pcs=np.asarray(Myotome_left_mesh.points),
    model2_pcs=np.asarray(Myotome_left_0_2_mesh.points),
)
print(score1)

score2 = st.tdr.pairwise_shape_similarity(
    model1_pcs=np.asarray(Myotome_right_mesh.points),
    model2_pcs=np.asarray(Myotome_right_0_2_mesh.points),
)
print(score2)


# 40%


In [ ]:
n_sample1 = 9315
n_sample2 = 10124

rng = np.random.default_rng(42)
idx1 = rng.choice(Myotome_left.n_obs, size=n_sample1, replace=False)
idx2 = rng.choice(Myotome_right.n_obs, size=n_sample2, replace=False)

Myotome_left_0_4 = Myotome_left[idx1].copy()
Myotome_right_0_4 = Myotome_right[idx2].copy()
Myotome_left_0_4, Myotome_right_0_4


In [ ]:
Myotome_left_0_4_pc, plot_cmap = st.tdr.construct_pc(
    adata=Myotome_left_0_4.copy(),
    spatial_key="spatial",
    groupby="celltype",
    key_added="tissue",
    colormap="#00BFC4",
)

Myotome_right_0_4_pc, plot_cmap = st.tdr.construct_pc(
    adata=Myotome_right_0_4.copy(),
    spatial_key="spatial",
    groupby="celltype",
    key_added="tissue",
    colormap="#00BFC4",
)


# Right-myotome mesh at 40% retained cells


In [ ]:
Myotome_right_0_4_mesh, _, _ = st.tdr.construct_surface(
    pc=Myotome_right_0_4_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.6},
    smooth=5000,
    scale_factor=1.00,
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([Myotome_right_0_4_mesh, Myotome_right_0_4_pc]),
    key="tissue",
    model_style=["surface", "points"],
    show_axes=True,
    jupyter="static",
    window_size=(800, 800),
    cpo=cpo1,
)


# Left-myotome mesh at 40% retained cells


In [ ]:
Myotome_left_0_4_mesh, _, _ = st.tdr.construct_surface(
    pc=Myotome_left_0_4_pc,
    key_added="tissue",
    alpha=0.6,
    cs_method="marching_cube",
    cs_args={"mc_scale_factor": 1.6},
    smooth=5000,
    scale_factor=1.00,
)


In [ ]:
st.pl.three_d_plot(
    model=st.tdr.collect_models([Myotome_left_0_4_mesh, Myotome_left_0_4_pc]),
    key="tissue",
    model_style=["surface", "points"],
    show_axes=True,
    jupyter="static",
    window_size=(800, 800),
    cpo=cpo1,
)


In [ ]:
score1 = st.tdr.pairwise_shape_similarity(
    model1_pcs=np.asarray(Myotome_left_mesh.points),
    model2_pcs=np.asarray(Myotome_left_0_4_mesh.points),
)

score2 = st.tdr.pairwise_shape_similarity(
    model1_pcs=np.asarray(Myotome_right_mesh.points),
    model2_pcs=np.asarray(Myotome_right_0_4_mesh.points),
)
print(score1)
print(score2)
